# Aula 24 — Gate II: experimento completo de Machine Learning clássico

Este notebook executa um capstone **didático** de classificação binária. Ele compara um
baseline, uma regressão logística e uma Random Forest por validação cruzada aninhada,
congela a decisão, escolhe o limiar somente com previsões *out-of-fold* do treino e abre
o teste externo uma única vez.

> **Aviso de escopo:** o dataset Wisconsin Breast Cancer (Diagnostic) é usado como
> benchmark educacional. Este artefato não foi validado para diagnóstico, triagem ou
> qualquer decisão clínica.

**Dependências mínimas:** Python 3.11, NumPy 1.26, pandas 2.1, Matplotlib 3.8,
scikit-learn 1.4 e nbformat 5.9. Não há download, segredo ou credencial.


## 1. Protocolo congelado antes do teste

- **Pergunta:** com as 30 medidas já presentes no benchmark, é possível discriminar
  exemplos rotulados como malignos melhor que a prevalência?
- **Unidade:** uma amostra digitalizada de aspirado por agulha fina.
- **Target positivo:** `1 = maligno`; `0 = benigno`.
- **Teste externo:** 20%, estratificado, separado antes de qualquer seleção.
- **Métrica primária:** ROC-AUC; secundárias: AP e Brier.
- **Candidatos:** Dummy, regressão logística e Random Forest.
- **Seleção:** validação cruzada aninhada no treino. Entre famílias a até um erro-padrão
  da melhor média externa, escolher a de menor complexidade.
- **Política:** no treino, escolher o maior limiar OOF com recall ≥ 95%; depois congelar.
- **Seed:** 20260909.


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260909
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 8)
pd.set_option("display.precision", 6)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit_learn": sklearn.__version__,
    "seed": SEED,
})


## 2. Snapshot, target e lacre do teste

`load_breast_cancer` distribui uma cópia versionada com o scikit-learn. O target original
usa `0 = malignant`; a expressão abaixo o transforma explicitamente em `1 = maligno`.
O hash cobre features, target e nomes de colunas. Os índices do teste são separados e
não reaparecem até a seção “abertura única”.


In [ ]:
raw = load_breast_cancer(as_frame=True)
X = raw.data.astype("float64")
y = (raw.target == 0).astype("int8").rename("maligno")
all_idx = np.arange(len(X))

train_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)
X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()
test_access_count = 0

snapshot_bytes = (
    X.to_csv(index=False, float_format="%.17g")
    + y.to_csv(index=False)
    + "|".join(X.columns)
).encode("utf-8")
data_sha256 = hashlib.sha256(snapshot_bytes).hexdigest()
split_sha256 = hashlib.sha256(
    np.asarray(train_idx, dtype="int64").tobytes()
    + np.asarray(test_idx, dtype="int64").tobytes()
).hexdigest()

assert len(X) == 569 and X.shape[1] == 30
assert set(train_idx).isdisjoint(set(test_idx))
assert len(train_idx) + len(test_idx) == len(X)
print({
    "shape_total": X.shape,
    "n_treino": len(train_idx),
    "n_teste_lacrado": len(test_idx),
    "prevalencia_total": round(float(y.mean()), 6),
    "prevalencia_treino": round(float(y_train.mean()), 6),
    "data_sha256_12": data_sha256[:12],
    "split_sha256_12": split_sha256[:12],
})


### Qualidade e unidade de análise

O benchmark não contém identificador de paciente nem atributos demográficos. Portanto,
podemos verificar completude, finitude e duplicatas exatas de features, mas **não**
deduplicar pessoas nem auditar desempenho por grupo social. Essa ausência é uma ameaça
à validade, não uma licença para declarar equidade.


In [ ]:
quality = {
    "nulos": int(X.isna().sum().sum()),
    "nao_finitos": int((~np.isfinite(X.to_numpy())).sum()),
    "linhas_feature_duplicadas": int(X.duplicated().sum()),
    "classes": y.value_counts().sort_index().to_dict(),
}
assert quality["nulos"] == 0
assert quality["nao_finitos"] == 0
assert set(quality["classes"]) == {0, 1}
print(quality)


## 3. Famílias, espaços de busca e nested CV

A escala é aprendida **dentro** do pipeline da logística. Cada fold externo simula dados
novos; dentro dele, quatro folds internos escolhem hiperparâmetros. A floresta recebe um
espaço pequeno e declarado, adequado ao objetivo pedagógico e ao orçamento de CPU.


In [ ]:
logistic = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        solver="liblinear", max_iter=5000, random_state=SEED
    )),
])
forest = Pipeline([
    ("model", RandomForestClassifier(
        n_estimators=120, max_features="sqrt", n_jobs=1, random_state=SEED
    )),
])

candidates = {
    "logistica": (
        logistic,
        {"model__C": [0.01, 0.1, 1.0, 10.0],
         "model__class_weight": [None, "balanced"]},
    ),
    "floresta": (
        forest,
        {"model__max_depth": [None, 6],
         "model__min_samples_leaf": [1, 3],
         "model__class_weight": [None, "balanced"]},
    ),
}
search_sizes = {name: int(np.prod([len(v) for v in grid.values()]))
                for name, (_, grid) in candidates.items()}
assert search_sizes == {"logistica": 8, "floresta": 8}
print({"candidatos_por_familia": search_sizes, "folds_externos": 5, "folds_internos": 4})


In [ ]:
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
best_params_by_fold = {name: [] for name in candidates}

for fold, (fit_pos, val_pos) in enumerate(outer.split(X_train, y_train), start=1):
    X_fit, X_val = X_train.iloc[fit_pos], X_train.iloc[val_pos]
    y_fit, y_val = y_train.iloc[fit_pos], y_train.iloc[val_pos]

    dummy = DummyClassifier(strategy="prior").fit(X_fit, y_fit)
    p_dummy = dummy.predict_proba(X_val)[:, 1]
    rows.append({
        "modelo": "dummy", "fold": fold,
        "roc_auc": roc_auc_score(y_val, p_dummy),
        "average_precision": average_precision_score(y_val, p_dummy),
        "brier": brier_score_loss(y_val, p_dummy),
    })

    inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED + fold)
    for name, (estimator, grid) in candidates.items():
        search = GridSearchCV(
            estimator, grid, scoring="roc_auc", cv=inner,
            n_jobs=1, refit=True, error_score="raise",
        )
        search.fit(X_fit, y_fit)
        p = search.predict_proba(X_val)[:, 1]
        rows.append({
            "modelo": name, "fold": fold,
            "roc_auc": roc_auc_score(y_val, p),
            "average_precision": average_precision_score(y_val, p),
            "brier": brier_score_loss(y_val, p),
        })
        best_params_by_fold[name].append(search.best_params_)

nested = pd.DataFrame(rows)
assert len(nested) == 15
assert nested.groupby("modelo")["fold"].nunique().eq(5).all()
print(nested.pivot(index="fold", columns="modelo", values="roc_auc").round(6))


## 4. Resultado do treino e regra de seleção

O desvio-padrão abaixo descreve variação entre os cinco folds externos; o erro-padrão
(`SE = s / √5`) é usado apenas na regra de parcimônia pré-declarada. Folds não são cinco
experimentos independentes, logo não tratamos esse resumo como teste de hipótese.


In [ ]:
summary = nested.groupby("modelo").agg(
    roc_auc_media=("roc_auc", "mean"),
    roc_auc_dp=("roc_auc", "std"),
    ap_media=("average_precision", "mean"),
    brier_medio=("brier", "mean"),
).sort_values("roc_auc_media", ascending=False)
summary["roc_auc_se"] = summary["roc_auc_dp"] / np.sqrt(5)

learners = summary.loc[["logistica", "floresta"]]
best_name = learners["roc_auc_media"].idxmax()
cutoff = learners.loc[best_name, "roc_auc_media"] - learners.loc[best_name, "roc_auc_se"]
eligible = learners.index[learners["roc_auc_media"] >= cutoff].tolist()
complexity_rank = {"logistica": 1, "floresta": 2}
chosen_family = min(eligible, key=lambda name: complexity_rank[name])

assert summary.loc[chosen_family, "roc_auc_media"] > summary.loc["dummy", "roc_auc_media"]
assert summary.loc[chosen_family, "ap_media"] > summary.loc["dummy", "ap_media"]
print(summary.round(6))
print({"melhor_media": best_name, "corte_1SE": round(float(cutoff), 6),
       "elegiveis": eligible, "familia_congelada": chosen_family})


In [ ]:
pivot = nested.pivot(index="fold", columns="modelo", values="roc_auc")
delta = pivot[chosen_family] - pivot["dummy"]
print({
    "ganho_auc_vs_dummy_medio": round(float(delta.mean()), 6),
    "ganho_min_fold": round(float(delta.min()), 6),
    "ganho_max_fold": round(float(delta.max()), 6),
    "fits_nested_planejados": 5 * 4 * sum(search_sizes.values()),
})
assert (delta > 0).all()


## 5. Refit no treino e limiar OOF

A família já está congelada. Agora a busca interna é repetida sobre todo o treino para
escolher sua configuração. Com essa configuração fixa, `cross_val_predict` gera uma
probabilidade para cada linha a partir de um modelo que não a viu. O maior limiar que
mantém recall OOF de pelo menos 95% é escolhido antes de tocar no teste.


In [ ]:
final_estimator, final_grid = candidates[chosen_family]
inner_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 100)
final_search = GridSearchCV(
    final_estimator, final_grid, scoring="roc_auc", cv=inner_full,
    n_jobs=1, refit=True, error_score="raise",
).fit(X_train, y_train)
best_model = final_search.best_estimator_

oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 200)
oof_p = cross_val_predict(
    clone(best_model), X_train, y_train, cv=oof_cv,
    method="predict_proba", n_jobs=1,
)[:, 1]

threshold_rows = []
for threshold in np.unique(np.r_[0.0, oof_p, 1.0]):
    pred = (oof_p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": float(threshold),
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "precision": tp / (tp + fp) if tp + fp else 0.0,
    })
threshold_table = pd.DataFrame(threshold_rows)
feasible = threshold_table.query("recall >= 0.95").sort_values(
    ["specificity", "precision", "threshold"], ascending=False
)
chosen_threshold = float(feasible.iloc[0]["threshold"])

assert not feasible.empty
assert 0.0 < chosen_threshold < 1.0
print({
    "melhores_parametros": final_search.best_params_,
    "auc_cv_interna_final": round(float(final_search.best_score_), 6),
    "auc_oof": round(float(roc_auc_score(y_train, oof_p)), 6),
    "limiar_congelado": round(chosen_threshold, 6),
    "recall_oof": round(float(feasible.iloc[0]["recall"]), 6),
    "especificidade_oof": round(float(feasible.iloc[0]["specificity"]), 6),
})


## 6. Abertura única do teste externo

Somente nesta célula os índices lacrados são materializados. O resultado serve para
estimar generalização da **decisão completa** — família, hiperparâmetros e limiar — e não
para iniciar outra rodada de tuning.


In [ ]:
assert test_access_count == 0
test_access_count += 1
X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()

test_p = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_p >= chosen_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()
specificity = tn / (tn + fp)
test_metrics = {
    "roc_auc": roc_auc_score(y_test, test_p),
    "average_precision": average_precision_score(y_test, test_p),
    "brier": brier_score_loss(y_test, test_p),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "specificity": specificity,
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred),
}

assert len(test_p) == len(test_idx)
assert np.all((test_p >= 0) & (test_p <= 1))
print(pd.Series(test_metrics).round(6))
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=["real_benigno", "real_maligno"],
                   columns=["pred_benigno", "pred_maligno"]))


### Incerteza no teste

O bootstrap reamostra as 114 unidades do teste com reposição. Amostras sem as duas
classes são descartadas para ROC-AUC. O intervalo percentil de 95% quantifica a
variabilidade de amostragem sob este conjunto; não cobre mudança de hospital, sensor,
época, prevalência ou protocolo de coleta.


In [ ]:
boot_rng = np.random.default_rng(SEED + 300)
boot = {"roc_auc": [], "average_precision": [], "brier": []}
for _ in range(2000):
    draw = boot_rng.integers(0, len(y_test), len(y_test))
    y_b = y_test.to_numpy()[draw]
    p_b = test_p[draw]
    if np.unique(y_b).size < 2:
        continue
    boot["roc_auc"].append(roc_auc_score(y_b, p_b))
    boot["average_precision"].append(average_precision_score(y_b, p_b))
    boot["brier"].append(brier_score_loss(y_b, p_b))

ci = {name: tuple(np.quantile(values, [0.025, 0.975]))
      for name, values in boot.items()}
assert min(map(len, boot.values())) >= 1900
print(pd.DataFrame(ci, index=["limite_2.5%", "limite_97.5%"]).T.round(6))


## 7. Curvas e análise de erros

ROC e Precision–Recall avaliam ranking em todos os limiares; a matriz resume a política
congelada. Os índices exibidos são posições no benchmark, não identificadores de pessoas.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_p)
pr_precision, pr_recall, _ = precision_recall_curve(y_test, test_p)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].plot(fpr, tpr, label=f"AUC = {test_metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="0.6")
axes[0].set(xlabel="Taxa de falsos positivos", ylabel="Recall", title="ROC — teste")
axes[0].legend()
axes[1].plot(pr_recall, pr_precision,
             label=f"AP = {test_metrics['average_precision']:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="0.6", label="prevalência")
axes[1].set(xlabel="Recall", ylabel="Precisão", title="Precision–Recall — teste")
axes[1].legend()
axes[2].imshow([[tn, fp], [fn, tp]], cmap="Blues")
for i, row in enumerate([[tn, fp], [fn, tp]]):
    for j, value in enumerate(row):
        axes[2].text(j, i, str(value), ha="center", va="center")
axes[2].set(xticks=[0, 1], yticks=[0, 1],
            xticklabels=["benigno", "maligno"], yticklabels=["benigno", "maligno"],
            xlabel="Predito", ylabel="Real", title="Matriz — limiar congelado")
plt.tight_layout()
plt.show()


In [ ]:
errors = pd.DataFrame({
    "indice_benchmark": test_idx,
    "real": y_test.to_numpy(),
    "prob_maligno": test_p,
    "predito": test_pred,
})
errors["tipo"] = np.select(
    [(errors.real == 1) & (errors.predito == 0),
     (errors.real == 0) & (errors.predito == 1)],
    ["falso_negativo", "falso_positivo"],
    default="acerto",
)
misclassified = errors.query("tipo != 'acerto'").copy()
misclassified["margem_ao_limiar"] = (
    misclassified["prob_maligno"] - chosen_threshold
).abs()

print(misclassified.sort_values("margem_ao_limiar").head(10).round(6))
print({"erros_por_tipo": misclassified["tipo"].value_counts().to_dict(),
       "n_erros": len(misclassified)})
assert len(misclassified) == fp + fn


## 8. Custo computacional e reprodutibilidade

A latência abaixo é uma micro-medição local, em lote e em memória. Ela não inclui rede,
serialização, monitoramento ou concorrência e, portanto, **não é SLA de produção**.
O refit idêntico testa determinismo nas condições desta execução.


In [ ]:
for _ in range(20):
    best_model.predict_proba(X_test)
samples_us = []
for _ in range(300):
    start = time.perf_counter_ns()
    best_model.predict_proba(X_test)
    samples_us.append((time.perf_counter_ns() - start) / 1_000 / len(X_test))
latency_us_per_row = float(np.median(samples_us))

refit = clone(best_model).fit(X_train, y_train)
refit_p = refit.predict_proba(X_test)[:, 1]
max_refit_diff = float(np.max(np.abs(test_p - refit_p)))

if chosen_family == "logistica":
    scaler = best_model.named_steps["scale"]
    max_scaler_diff = float(np.max(np.abs(scaler.mean_ - X_train.mean().to_numpy())))
    assert max_scaler_diff < 1e-12
else:
    max_scaler_diff = float("nan")
assert max_refit_diff < 1e-12
print({"latencia_mediana_us_por_linha": round(latency_us_per_row, 6),
       "diferenca_maxima_refit": max_refit_diff,
       "erro_media_scaler_vs_treino": max_scaler_diff})


In [ ]:
config = {
    "seed": SEED,
    "test_fraction": 0.20,
    "outer_folds": 5,
    "inner_folds": 4,
    "primary_metric": "roc_auc",
    "threshold_constraint": "OOF recall >= 0.95",
    "chosen_family": chosen_family,
    "best_params": final_search.best_params_,
    "chosen_threshold": chosen_threshold,
}
config_sha256 = hashlib.sha256(
    json.dumps(config, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()
prediction_sha256 = hashlib.sha256(
    np.asarray(test_p, dtype="float64").tobytes()
).hexdigest()
manifest = {
    "data_sha256": data_sha256,
    "split_sha256": split_sha256,
    "config_sha256": config_sha256,
    "prediction_sha256": prediction_sha256,
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
}
experiment_id = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(json.dumps({"experiment_id": experiment_id, **manifest}, indent=2, ensure_ascii=False))


## 9. Auditoria automática do Gate II

As verificações não provam validade externa, mas tornam falhas mecânicas e metodológicas
observáveis. Uma nota alta depende tanto dos checks quanto de uma discussão honesta das
ameaças à validade.


In [ ]:
gate_checks = {
    "pergunta_target_unidade": True,
    "teste_externo_disjunto": set(train_idx).isdisjoint(set(test_idx)),
    "baseline_explicito": "dummy" in summary.index,
    "preprocessamento_em_pipeline": isinstance(logistic, Pipeline),
    "cv_aninhada": nested.groupby("modelo")["fold"].nunique().min() == 5,
    "abertura_unica_do_teste": test_access_count == 1 and len(test_idx) == len(test_p),
    "metricas_justificadas": {"roc_auc", "average_precision", "brier"}.issubset(test_metrics),
    "limiar_por_oof": recall_score(y_train, oof_p >= chosen_threshold) >= 0.95,
    "incerteza_reportada": all(len(v) >= 1900 for v in boot.values()),
    "erros_analisados": len(misclassified) == fp + fn,
    "provenance": all(key in manifest for key in
                      ["data_sha256", "split_sha256", "config_sha256", "prediction_sha256"]),
    "determinismo": max_refit_diff < 1e-12,
}
assert all(gate_checks.values())
print(pd.Series(gate_checks, name="aprovado"))
print(f"Gate II: {sum(gate_checks.values())}/{len(gate_checks)} verificações aprovadas")


## 10. O que o experimento sustenta — e o que não sustenta

**Sustenta:** neste snapshot, sob divisão aleatória estratificada e no espaço de busca
declarado, a família selecionada supera o baseline e mantém forte discriminação no teste
externo. O manifesto liga dados, split, configuração, ambiente e previsões.

**Não sustenta:** eficácia clínica; causalidade; equidade entre grupos; transferência para
outro hospital, sensor, período ou prevalência; ausência de duplicatas de pacientes; ou
que nenhum modelo fora da busca seria melhor. A amostra de teste é pequena, o benchmark
é antigo e a divisão aleatória não simula mudança temporal/institucional.

## Exercícios de transferência

1. Troque o requisito OOF para recall ≥ 99% e explique a mudança em falsos positivos.
2. Use `GroupKFold` supondo que várias linhas possam pertencer à mesma pessoa. Qual
   identificador faltaria para tornar a análise válida?
3. Acrescente custo de falso negativo cinco vezes maior e derive a utilidade do limiar.
4. Faça uma análise de sensibilidade com três seeds **sem reabrir o teste para escolher**.
5. Escreva um model card curto que proíba uso clínico e declare a população do benchmark.

### Respostas esperadas, em síntese

1. A restrição mais forte tende a baixar o limiar e elevar falsos positivos.
2. Seria necessário um ID de paciente confiável; sem ele, a independência não é auditável.
3. Minimize `5·FN + 1·FP` no OOF, nunca no teste.
4. Reavalie a estabilidade no treino/CV e mantenha a decisão externa lacrada.
5. Inclua propósito, dados, métricas, riscos, usos proibidos, proveniência e monitoramento.

## Próximo passo

No M5, começaremos **redes neurais do zero**. A Aula 01 decompõe um neurônio artificial
em `z = XW + b`, ativação e loss — preservando a mesma disciplina experimental deste gate.
